In [1]:
import ultralytics
import cv2
from ultralytics import solutions

In [12]:
import cv2
from ultralytics import solutions

# -----------------------------
# Video Input
# -----------------------------
cap = cv2.VideoCapture(r"D:/TE/Internship/code/data/raw_videos/elephant.mp4")

# Get original properties
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

print("Original Resolution:", w, "x", h)

# -----------------------------
# Resize Settings (for display)
# -----------------------------
display_width = 960
display_height = 540

# -----------------------------
# Video Writer (save original size)
# -----------------------------
video_writer = cv2.VideoWriter(
    r"D:/TE/Internship/code/data/processed_videos/elephant.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

# -----------------------------
# Region for Counting
# -----------------------------
region_points = [
    (0, 0),
    (w - 1, 0),
    (w - 1, h - 1),
    (0, h - 1)
]

# -----------------------------
# Object Counter
# -----------------------------
counter = solutions.ObjectCounter(
    show=False,  # Disable automatic large window
    region=region_points,
    model=r"D:/TE/Internship/code/models/trained/best_10000_images.pt"
)

# -----------------------------
# Processing Loop
# -----------------------------
cv2.namedWindow("Output", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Output", display_width, display_height)

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # Run counting
    results = counter(frame)

    # Write original size frame to video
    video_writer.write(results.plot_im)

    # Resize only for display
    display_frame = cv2.resize(results.plot_im, (display_width, display_height))
    cv2.imshow("Output", display_frame)

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# -----------------------------
# Release Everything
# -----------------------------
cap.release()
video_writer.release()
cv2.destroyAllWindows()

Original Resolution: 1920 x 1080
Ultralytics Solutions:  {'source': None, 'model': 'D:/TE/Internship/code/models/trained/best_10000_images.pt', 'classes': None, 'show_conf': True, 'show_labels': True, 'region': [(0, 0), (1919, 0), (1919, 1079), (0, 1079)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
0: 1080x1920 0.8ms, 1 Elephant
Speed: 236.7ms track, 0.8ms solution per image at shape (1, 3, 1080, 1920)

1: 1080x1920 0.7ms, 1 Elephant
Speed: 138.3ms track, 0.7ms solution per image at shape (1, 3, 1080, 1920)

2: 1080x1920 0.5ms, 1

KeyboardInterrupt: 

In [14]:
from ultralytics import YOLO
from collections import defaultdict
import cv2

model = YOLO("D:/TE/Internship/code/models/trained/best_10000_images.pt")

cap = cv2.VideoCapture("D:/TE/Internship/code/data/raw_videos/elephant.mp4")

unique_ids = defaultdict(set)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes
        ids = boxes.id.cpu().numpy()
        classes = boxes.cls.cpu().numpy()

        for obj_id, cls_id in zip(ids, classes):
            class_name = model.names[int(cls_id)]
            unique_ids[class_name].add(int(obj_id))

cap.release()

print("\nFinal Unique Animal Count:")
for species, ids in unique_ids.items():
    print(f"{species}: {len(ids)}")


0: 384x640 1 Elephant, 21.5ms
Speed: 4.4ms preprocess, 21.5ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 20.9ms
Speed: 2.1ms preprocess, 20.9ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 17.0ms
Speed: 2.0ms preprocess, 17.0ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 16.9ms
Speed: 1.9ms preprocess, 16.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 17.1ms
Speed: 1.9ms preprocess, 17.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 17.1ms
Speed: 1.9ms preprocess, 17.1ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Elephant, 16.9ms
Speed: 2.0ms preprocess, 16.9ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 Hippopotamus, 16.9ms
Speed: 1.9ms preprocess, 16.9ms inference, 0.7ms postprocess per imag

KeyboardInterrupt: 